In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'e:\\Mental Health Chatbot\\AI_chatbot'

In [3]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

e:\Mental Health Chatbot\AI_chatbot\SoulTalk\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#extract text from PDF files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [5]:
extracted_data = load_pdf_files("data")

In [6]:
extracted_data


[Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2015-10-28T17:52:55+00:00', 'moddate': '2015-10-28T17:53:18+00:00', 'trapped': '/False', 'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf', 'total_pages': 48, 'page': 0, 'page_label': '1'}, page_content='Version Six\nThe Little Book of \nMental Health\nA practical guide for\nEveryday Emotional Wellbeing'),
 Document(metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2015-10-28T17:52:55+00:00', 'moddate': '2015-10-28T17:53:18+00:00', 'trapped': '/False', 'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf', 'total_pages': 48, 'page': 1, 'page_label': '2'}, page_content='Sixth Edition - 2015\n© Primary Mental Health Service/Gloucestershire Health and \n Social Care Community have kindly given NHS Somerset \n permission to adapt this booklet from the original version \

In [7]:
#minimizing the document by removing unwanted data

from typing import List 
from langchain.schema import Document

def filter_to_minimal_docs(docs : List[Document])->List[Document]:
    """
    Given a list of Document objects , return a new list of Document objects 
    containing only 'source' in metadata and the original page_content.
    """

    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
            )
        )
    return minimal_docs


In [8]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [9]:
minimal_docs

[Document(metadata={'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf'}, page_content='Version Six\nThe Little Book of \nMental Health\nA practical guide for\nEveryday Emotional Wellbeing'),
 Document(metadata={'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf'}, page_content='Sixth Edition - 2015\n© Primary Mental Health Service/Gloucestershire Health and \n Social Care Community have kindly given NHS Somerset \n permission to adapt this booklet from the original version \n complied by Alison Sedgwick-Taylor – Clinical Psychologist, \n Primary Mental Health Service, Gloucestershire NHS \n Partnership Trust.\nOriginal design by: Hobbs Design Ltd.\nAdaptions to this version made by:\nAni Overton Design and Graphics 01258 830 361\nPaper produced from 100% recycled fibre and vegetable ink.\nDisclaimer:\nIn producing this booklet Somerset County Council, Public \nHealth has made every effort to provide advice based on up to \ndate evidence for what

In [10]:
#Split the documents into smaller chunks

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap = 20, 
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [11]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks :{len(texts_chunk)}")

Number of chunks :1198


In [12]:
texts_chunk

[Document(metadata={'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf'}, page_content='Version Six\nThe Little Book of \nMental Health\nA practical guide for\nEveryday Emotional Wellbeing'),
 Document(metadata={'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf'}, page_content='Sixth Edition - 2015\n© Primary Mental Health Service/Gloucestershire Health and \n Social Care Community have kindly given NHS Somerset \n permission to adapt this booklet from the original version \n complied by Alison Sedgwick-Taylor – Clinical Psychologist, \n Primary Mental Health Service, Gloucestershire NHS \n Partnership Trust.\nOriginal design by: Hobbs Design Ltd.\nAdaptions to this version made by:\nAni Overton Design and Graphics 01258 830 361'),
 Document(metadata={'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf'}, page_content='Paper produced from 100% recycled fibre and vegetable ink.\nDisclaimer:\nIn producing this booklet Somerset C

In [13]:
# downloading the embedding model

from langchain.embeddings import HuggingFaceBgeEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceBgeEmbeddings(
        model_name = model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\91828\AppData\Local\Temp\ipykernel_6672\2574500786.py:10: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(


In [14]:
embedding

HuggingFaceBgeEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_instruction='Represent this question for searching relevant passages: ', embed_instruction='', show_progress=False)

In [15]:
vector = embedding.embed_query("Hello world")
vector

[-0.010300801135599613,
 0.18307925760746002,
 0.030811261385679245,
 0.0044529009610414505,
 -0.027336183935403824,
 -0.03356251120567322,
 0.03763154149055481,
 -0.031573329120874405,
 -0.0033910225611180067,
 -0.008950827643275261,
 0.03803611919283867,
 -0.05129104107618332,
 0.0003682424430735409,
 -0.023727117106318474,
 0.09271018952131271,
 -0.027795882895588875,
 -0.03515256941318512,
 -0.00322411279194057,
 -0.07681780308485031,
 -0.057612109929323196,
 0.07257598638534546,
 0.11128547787666321,
 0.01605845056474209,
 0.01590847596526146,
 -0.08232694864273071,
 0.007007298059761524,
 0.02901308238506317,
 0.0011386480182409286,
 0.11671745032072067,
 -0.03232734650373459,
 -0.032271675765514374,
 -0.0012590596452355385,
 0.10591618716716766,
 0.023600798100233078,
 0.009664909914135933,
 0.09834078699350357,
 0.04293639957904816,
 -0.019547639414668083,
 0.019267937168478966,
 -0.06417104601860046,
 0.02392338588833809,
 -0.0528799444437027,
 -0.026469575241208076,
 0.005548

In [16]:
print("vector length:",len(vector))

vector length: 384


In [22]:
from dotenv import load_dotenv
import os
load_dotenv()
print(os.getenv("PINECONE_API_KEY"))

pcsk_3K4CNp_HT4G67nHGDJbvssyaTNydtNBSJMGCPfsNptKAiQFhhM4e4n3twYtRa6wT2TbEsb


In [23]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [24]:
from pinecone import Pinecone

pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)


In [25]:
pc


In [27]:
#creating databases for the chunks which is known as index

from pinecone import ServerlessSpec

index_name = "soultalk"

existing_indexes = [index["name"] for index in pc.list_indexes()]
    
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384, # dimensions of the embeddings
        metric="cosine", # Cosine similarity
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [28]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [29]:
# load existing index

from langchain_pinecone import PineconeVectorStore
#Embed each chunk and upsert the embeddings into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [30]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [31]:
retriever_docs = retriever.invoke("what is Acne")
retriever_docs

[Document(id='49c2c086-7e58-451e-bfc3-9ef46411157a', metadata={'source': 'data\\you-will-get-through-this-night-0063053888-9780063053885_compress.pdf'}, page_content='Chr onic  – long-term stress that sticks around, particularly if you’re always\nexposed to the problem (ongoing ﬁnancial problems, toxic\nenvironments/discrimination)\nAssess the stressor\nStressors are the things that cause stress, in case you didn’t guess. These\nthings include situations that change our regular way of doing things, that\nbring increased scrutiny from other people – but can also be things that\nshould be considered ‘good’, like moving house, getting married, or starting'),
 Document(id='64266235-4a06-499d-9eb8-ad2b9c7ae48e', metadata={'source': 'data\\The_little_book_of_mental_health_-_version_6_copy.pdf'}, page_content='Reading’ on page 38 and the list of  useful \norganisations and websites at the back of  this \nbooklet.'),
 Document(id='983f8b06-a888-4309-b522-931384c75bc6', metadata={'source': 'dat

In [32]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0
    )

In [33]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [34]:
system_prompt = (
    "You are SoulTalk, a kind, supportive, and empathetic AI companion designed to help people who may be struggling with their thoughts, emotions, stress, anxiety, loneliness, or other mental health challenges."
    "Your role is to speak like a caring friend and well-wisher who genuinely wants to help the user feel understood, supported, and less alone."

    "Guidelines for your behavior : "
    "Always respond with empathy, kindness, and patience."
    "Make the user feel heard and validated. Acknowledge their feelings before giving suggestions."
    "Speak in a warm, friendly, and natural conversational tone — like a trusted friend."
    "Encourage positive thinking, self-care, and small healthy steps that may help the user feel better."
    "When appropriate, gently motivate the user and remind them of their strengths and resilience."
    "If the user shares something difficult, respond calmly and compassionately without judging them."
    "Ask supportive follow-up questions when it can help the user express their feelings better."
    "Never shame, criticize, or invalidate the user emotions."

    "Important safety rules:"
    "You are NOT a doctor, therapist, or medical professional."
    "Do NOT diagnose mental illnesses."
    "Do NOT provide medical treatment or medication advice."
    "If the user seems to be in serious distress or mentions self-harm or suicide, encourage them to reach out to a trusted person, family member, or a mental health professional."
    
    "Conversation style:"
    "Use simple and comforting language."
    "Keep responses supportive but not overly long."
    "Sometimes include gentle encouragement or motivation."
    "Make the user feel safe to talk openly."

    "Your goal is to help the user feel supported, understood, and hopeful while providing thoughtful responses to their concerns."

    "Use the following pieces of retreived context to answer"
    "and answer should be of maximum 30-40 words"
    "If you dont'know the answer, say that sorry! you dont have idea about this"
    ". Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)

In [35]:
question_answer_chain = create_stuff_documents_chain(chatModel,prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [36]:
response = rag_chain.invoke({"input":"hello"})
print(response["answer"])

Hello! It's so great to connect with you. I'm here to listen and support you in any way I can. How are you feeling today? Is there something on your mind that you'd like to talk about?


In [37]:
response = rag_chain.invoke({"input":"i am so sad"})
print(response["answer"])

Sweetheart, it's okay to feel sad. It's a normal emotion, and it doesn't mean you're weak or flawed. Can you tell me a little bit more about what's making you feel this way?
